In [1]:
import os
import gc
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error

from lgbm_model import LGBM_model

In [ ]:
# ====== Path ======
DATA_PATH = "dt_lgbm.parquet"
MODEL_DIR = Path("./artifacts_gru")
MODEL_DIR.mkdir(exist_ok=True)

# ====== Reproducibility ======
SEED = 42

# ====== GRU config ======
SEQ_LEN = 6
LABEL_COL = "vwap_rk"   
DATE_COL = "date"
STOCK_COL = "stkcd"

PROJ_DIM = 256
GRU_HIDDEN = 128
GRU_LAYERS = 2
DROPOUT = 0.2

BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4

# ====== Time segments ======
segments = {
    'train': slice('2017-08-01', '2020-07-31'),
    'valid': slice('2020-08-01', '2021-12-31'),
    'test':  slice('2022-01-01', '2025-08-01')
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
dt_lgbm = pd.read_parquet(DATA_PATH).copy()

dt_lgbm[DATE_COL] = pd.to_datetime(dt_lgbm[DATE_COL])

dt_lgbm = dt_lgbm.sort_values([DATE_COL, STOCK_COL])
dt_lgbm = dt_lgbm.drop_duplicates(subset=[DATE_COL, STOCK_COL], keep="first")

dt_lgbm = dt_lgbm.set_index([DATE_COL, STOCK_COL]).sort_index()

print(dt_lgbm.shape)
print(dt_lgbm.index.names)
print(dt_lgbm.head(3))

(11150, 4098)
['date', 'stkcd']
                      vwap_ret20   vwap_rk     emb_0     emb_1     emb_2  \
date       stkcd                                                           
2017-02-03 000078.SZ    0.068016  0.756349 -0.001916 -0.020751 -0.017450   
           000150.SZ   -0.021359 -1.388604 -0.009593 -0.019066  0.029564   
           000423.SZ    0.012821 -0.667631 -0.015075 -0.038073  0.026012   

                         emb_3     emb_4     emb_5     emb_6     emb_7  ...  \
date       stkcd                                                        ...   
2017-02-03 000078.SZ -0.020516  0.008843  0.019808 -0.000892 -0.006514  ...   
           000150.SZ -0.007572  0.014541  0.041269  0.010317  0.028720  ...   
           000423.SZ -0.004020  0.012119  0.023057  0.046349  0.030033  ...   

                      emb_4086  emb_4087  emb_4088  emb_4089  emb_4090  \
date       stkcd                                                         
2017-02-03 000078.SZ  0.012970 -0.017450  0

In [5]:
emb_cols = [c for c in dt_lgbm.columns if c.startswith("emb_")]
print("num emb cols:", len(emb_cols))
print("label cols exists:", [c for c in ["vwap_ret20", "vwap_rk"] if c in dt_lgbm.columns])

def cross_sectional_zscore(df, cols):
    df = df.copy()
    grouped = df.groupby(level=0)  # level 0 = date
    mean = grouped[cols].transform("mean")
    std = grouped[cols].transform("std").replace(0, np.nan)
    df[cols] = (df[cols] - mean) / (std + 1e-8)
    df[cols] = df[cols].fillna(0.0)
    return df

dt_lgbm = cross_sectional_zscore(dt_lgbm, emb_cols)

num emb cols: 4096
label cols exists: ['vwap_ret20', 'vwap_rk']


In [6]:
gru_train_slice = segments["train"]
gru_valid_slice = segments["valid"]

In [7]:
def build_sequence_samples(
    df: pd.DataFrame,
    feature_cols,
    label_col="vwap_rk",
    seq_len=6,
):
    """
    输入:
        df: MultiIndex [date, stkcd]
    输出:
        X: (N, seq_len, input_dim)
        y: (N,)
        meta: DataFrame columns=[date, stkcd]
    """
    df_reset = df.reset_index().sort_values([STOCK_COL, DATE_COL]).copy()

    X_list, y_list, meta_list = [], [], []

    for stk, g in tqdm(df_reset.groupby(STOCK_COL), desc="Building sequences"):
        g = g.sort_values(DATE_COL).reset_index(drop=True)

        feats = g[feature_cols].values.astype(np.float32)
        labels = g[label_col].values.astype(np.float32)
        dates = g[DATE_COL].values

        if len(g) < seq_len:
            continue

        for i in range(seq_len - 1, len(g)):
            x_seq = feats[i - seq_len + 1 : i + 1]
            y = labels[i]
            date = dates[i]

            if np.isnan(y):
                continue
            if np.isnan(x_seq).any():
                continue

            X_list.append(x_seq)
            y_list.append(y)
            meta_list.append((pd.Timestamp(date), stk))

    X = np.stack(X_list)
    y = np.array(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list, columns=[DATE_COL, STOCK_COL])

    return X, y, meta

In [8]:
X_all, y_all, meta_all = build_sequence_samples(
    dt_lgbm,
    feature_cols=emb_cols,
    label_col=LABEL_COL,
    seq_len=SEQ_LEN,
)

print("X_all:", X_all.shape)
print("y_all:", y_all.shape)
print(meta_all.head())

Building sequences: 100%|██████████| 430/430 [00:02<00:00, 204.96it/s]


X_all: (9237, 6, 4096)
y_all: (9237,)
        date      stkcd
0 2017-09-01  000028.SZ
1 2017-10-09  000028.SZ
2 2017-11-01  000028.SZ
3 2017-12-01  000028.SZ
4 2018-01-02  000028.SZ


In [9]:
def mask_by_slice(meta_df, date_slice):
    start = pd.Timestamp(date_slice.start)
    end = pd.Timestamp(date_slice.stop)
    return (meta_df[DATE_COL] >= start) & (meta_df[DATE_COL] <= end)

train_mask = mask_by_slice(meta_all, gru_train_slice)
valid_mask = mask_by_slice(meta_all, gru_valid_slice)

X_train = X_all[train_mask.values]
y_train = y_all[train_mask.values]

X_valid = X_all[valid_mask.values]
y_valid = y_all[valid_mask.values]

meta_train = meta_all.loc[train_mask].reset_index(drop=True)
meta_valid = meta_all.loc[valid_mask].reset_index(drop=True)

print(X_train.shape, y_train.shape)
print(X_valid.shape, y_valid.shape)

(2365, 6, 4096) (2365,)
(1395, 6, 4096) (1395,)


In [10]:
class SeqDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]

train_ds = SeqDataset(X_train, y_train)
valid_ds = SeqDataset(X_valid, y_valid)
all_ds = SeqDataset(X_all, None)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
all_loader = DataLoader(all_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

In [11]:
class GRUFeatureExtractor(nn.Module):
    def __init__(self, input_dim, proj_dim=256, gru_hidden=128, gru_layers=2, dropout=0.2):
        super().__init__()
        self.projection = nn.Linear(input_dim, proj_dim)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(proj_dim)

        self.gru = nn.GRU(
            input_size=proj_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0
        )
        self.pred_head = nn.Linear(gru_hidden, 1)

    def forward(self, x, extract_features=False):
        # x: (B, T, D)
        x = self.projection(x)
        x = self.act(x)
        x = self.norm(x)

        _, h_n = self.gru(x)      # h_n: (num_layers, B, hidden)
        feat = h_n[-1]            # (B, hidden)

        if extract_features:
            return feat

        pred = self.pred_head(feat).squeeze(-1)
        return pred

In [12]:
def run_epoch(model, loader, optimizer=None, device="cuda"):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    criterion = nn.MSELoss()

    total_loss = 0.0
    n = 0

    for batch in loader:
        if train_mode:
            x, y = batch
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        else:
            x, y = batch
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)

        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / n

def train_gru_model(model, train_loader, valid_loader, device="cuda",
                    epochs=20, lr=1e-3, weight_decay=1e-4, patience=4,
                    ckpt_path="artifacts_gru/best_gru.pt"):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_valid = float("inf")
    bad_epochs = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer=optimizer, device=device)
        valid_loss = run_epoch(model, valid_loader, optimizer=None, device=device)

        history.append((epoch, train_loss, valid_loss))
        print(f"Epoch {epoch:02d} | train={train_loss:.6f} | valid={valid_loss:.6f}")

        if valid_loss < best_valid:
            best_valid = valid_loss
            bad_epochs = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    return model, pd.DataFrame(history, columns=["epoch", "train_loss", "valid_loss"])

In [13]:
input_dim = X_all.shape[-1]

gru_model = GRUFeatureExtractor(
    input_dim=input_dim,
    proj_dim=PROJ_DIM,
    gru_hidden=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

gru_model, train_hist = train_gru_model(
    gru_model,
    train_loader,
    valid_loader,
    device=DEVICE,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    ckpt_path=str(MODEL_DIR / "best_gru.pt")
)

train_hist

Epoch 01 | train=1.100384 | valid=1.334179
Epoch 02 | train=0.989421 | valid=1.448125
Epoch 03 | train=0.912370 | valid=1.489183
Epoch 04 | train=0.811610 | valid=1.506371
Epoch 05 | train=0.693054 | valid=1.455406
Early stopping triggered.


C:\Users\zexua\AppData\Local\Temp\ipykernel_35968\1140557508.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(ckpt_path, map_location=de

,epoch,train_loss,valid_loss
0,1,1.100384,1.334179
1,2,0.989421,1.448125
2,3,0.912370,1.489183
3,4,0.811610,1.506371
4,5,0.693054,1.455406


In [14]:
def extract_features(model, loader, device="cuda"):
    model.eval()
    feats = []

    with torch.no_grad():
        for x in tqdm(loader, desc="Extracting GRU features"):
            x = x.to(device, non_blocking=True)
            f = model(x, extract_features=True)
            feats.append(f.detach().cpu().numpy())

    feats = np.concatenate(feats, axis=0)
    return feats

gru_feats = extract_features(gru_model, all_loader, device=DEVICE)
print(gru_feats.shape)

Extracting GRU features: 100%|██████████| 37/37 [00:00<00:00, 72.68it/s]

(9237, 128)


In [15]:
gru_cols = [f"gru_{i}" for i in range(gru_feats.shape[1])]

df_gru_features = pd.DataFrame(gru_feats, columns=gru_cols)
df_gru_features = pd.concat([meta_all.reset_index(drop=True), df_gru_features], axis=1)
df_gru_features[LABEL_COL] = y_all

df_gru_features = df_gru_features.set_index([DATE_COL, STOCK_COL]).sort_index()

print(df_gru_features.shape)
print(df_gru_features.head())

(9237, 129)
                         gru_0     gru_1     gru_2     gru_3     gru_4  \
date       stkcd                                                         
2017-07-03 000423.SZ -0.039820  0.404210  0.079866 -0.096858  0.118615   
           000513.SZ -0.393639  0.344916  0.093672  0.017725 -0.401961   
           000538.SZ -0.193043  0.100295  0.283238  0.363118  0.113822   
           000661.SZ -0.135889 -0.011003  0.131026 -0.046144 -0.409539   
           002019.SZ  0.612689  0.099370 -0.352351  0.037285  0.130063   

                         gru_5     gru_6     gru_7     gru_8     gru_9  ...  \
date       stkcd                                                        ...   
2017-07-03 000423.SZ -0.063481 -0.734158 -0.413243 -0.312607 -0.202994  ...   
           000513.SZ -0.648525 -0.421016 -0.181629  0.003193  0.066030  ...   
           000538.SZ  0.262718  0.362793  0.051354  0.284189  0.546002  ...   
           000661.SZ  0.243628  0.434776 -0.538486 -0.323478 -0.162542  ..

In [16]:
feature_cols = [c for c in df_gru_features.columns if c.startswith("gru_")]
label_col = LABEL_COL

print(len(feature_cols), label_col)

128 vwap_rk


In [17]:
base_params = {
    "objective": "regression",
    "metric": "l2",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "min_data_in_leaf": 50,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,
}

evals_result = {}
lgbm = LGBM_model(
    loss="mse",
    early_stopping_rounds=50,
    num_boost_round=1000,
    **base_params
)

lgbm.fit(
    dataset=df_gru_features,
    segments=segments,
    feature_cols=feature_cols,
    label_col=label_col,
    verbose_eval=50,
    evals_result=evals_result
)

train_pred, valid_pred, test_pred = lgbm.predict(
    dataset=df_gru_features,
    segments=segments,
    feature_cols=feature_cols
)

print("train_pred:", train_pred.shape)
print("valid_pred:", valid_pred.shape)
print("test_pred:", test_pred.shape)

Training until validation scores don't improve for 50 rounds
[50]	train's l2: 0.648142	valid's l2: 1.42728
train_pred: (2365,)
valid_pred: (1395,)
test_pred: (5462,)


In [18]:
def mse_on_segment(pred_series, dataset, label_col):
    true = dataset.loc[pred_series.index, label_col]
    return mean_squared_error(true.values, pred_series.values)

print("Train MSE:", mse_on_segment(train_pred, df_gru_features, label_col))
print("Valid MSE:", mse_on_segment(valid_pred, df_gru_features, label_col))
print("Test  MSE:", mse_on_segment(test_pred,  df_gru_features, label_col))

Train MSE: 1.0516339823897483
Valid MSE: 1.3085110980491355
Test  MSE: 1.202001022994957


In [19]:
test_factor = test_pred.rename("gru_lgbm_factor").reset_index()
test_factor[DATE_COL] = pd.to_datetime(test_factor[DATE_COL]).dt.strftime("%Y%m%d")

test_factor.to_csv(MODEL_DIR / "gru_lgbm_test_factor_long.csv", index=False)

# 如果你想要宽表
test_factor_wide = test_factor.pivot(index=DATE_COL, columns=STOCK_COL, values="gru_lgbm_factor")
test_factor_wide.to_csv(MODEL_DIR / "gru_lgbm_test_factor_wide.csv")

print("saved:", MODEL_DIR / "gru_lgbm_test_factor_long.csv")
print("saved:", MODEL_DIR / "gru_lgbm_test_factor_wide.csv")

saved: artifacts_gru\gru_lgbm_test_factor_long.csv
saved: artifacts_gru\gru_lgbm_test_factor_wide.csv
